# 1. 환경·설정

Core-4 Pilot Probe: **2023-12-31–2025-12-31, 양끝 포함**. 승인: ADR-003.
현재 조회 데이터의 접근·스키마·단위·기초 품질을 검사한다. 과거 빈티지 재현이나 학습 데이터 완성을 주장하지 않는다.

저장소 기존 .venv의 Coffee Probe kernel을 선택하고 Restart Kernel → Run All로 실행한다.
첫 셀에서 실제 Python 경로를 확인한다. 매 전체 실행은 새로운 run_id를 생성한다.
인증값은 출력하지 않으며 NASA_API_KEY를 요청에 사용하지 않는다.
원자료·library 반환값·검증 CSV·시스템 메타데이터를 분리한다. Mock/보간/ffill/이상치 삭제/롤 보정은 없다.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
from itertools import combinations
from io import BytesIO, StringIO
from zipfile import ZipFile
from urllib.parse import quote, urlsplit
import contextlib
import hashlib
import json
import logging
import os
import platform
import re
import sys
import tempfile
import time
import uuid

import numpy as np
import pandas as pd
import requests
import yfinance as yf
from dotenv import load_dotenv
from IPython.display import display

def find_repository_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "Context.md").is_file() and (candidate / ".git").is_dir():
            return candidate
    raise RuntimeError("repository_root_not_found")

ROOT = find_repository_root()
load_dotenv(ROOT / ".env", override=False)
FRED_KEY = os.environ.get("FRED_API_KEY", "").strip()
# Only used for redaction; never passed as a NASA credential.
_SECRET_VALUES = [value for name, value in os.environ.items()
                  if re.search(r"(API_KEY|TOKEN|PASSWORD|SECRET)$", name, re.I) and value]
_SECRET_VALUES = sorted(set(_SECRET_VALUES + ([FRED_KEY] if FRED_KEY else [])), key=len, reverse=True)

START = pd.Timestamp("2023-12-31")
END = pd.Timestamp("2025-12-31")
EXPECTED_DAYS = pd.date_range(START, END, freq="D")
REGIONS = [
    dict(region_id="br_sul_minas", country="Brazil", region="Sul de Minas", point="Três Pontas", latitude=-21.3700, longitude=-45.5100),
    dict(region_id="br_cerrado", country="Brazil", region="Cerrado Mineiro", point="Patrocínio", latitude=-18.9439, longitude=-46.9925),
    dict(region_id="br_alta_mogiana", country="Brazil", region="Alta Mogiana", point="Franca", latitude=-20.5389, longitude=-47.4008),
    dict(region_id="co_huila", country="Colombia", region="Huila", point="Gigante / Jorge Villamil", latitude=2.3333, longitude=-75.5167),
    dict(region_id="co_caldas", country="Colombia", region="Caldas", point="Chinchiná / Naranjal", latitude=4.9667, longitude=-75.6500),
    dict(region_id="co_antioquia", country="Colombia", region="Antioquia", point="Venecia / El Rosario", latitude=5.9667, longitude=-75.7000),
]
WEATHER_VARS = ["PRECTOTCORR", "T2M", "T2M_MIN", "T2M_MAX", "RH2M"]
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + "_" + uuid.uuid4().hex[:8]
RUN_DIR = ROOT / "data" / "raw" / "probes" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
ARTIFACTS, REQUESTS, RESULTS = [], [], []
WEATHER_TABLES, COT_PARTS, COT_ARCHIVES = {}, [], []
SESSION = requests.Session()
SESSION.trust_env = False
SESSION.headers["User-Agent"] = "CoffeePriceResearchProbe/1.0"
MAX_ATTEMPTS = 3
HTTP_TIMEOUT = (10, 45)
_YF_CACHE = tempfile.TemporaryDirectory(prefix="coffee-probe-yfinance-")
yf.set_tz_cache_location(_YF_CACHE.name)
logging.getLogger("yfinance").setLevel(logging.CRITICAL)

ENVIRONMENT = {
    "python_version": platform.python_version(),
    "python_executable": sys.executable,
    "implementation": sys.implementation.name,
    "platform": platform.platform(),
    "machine": platform.machine(),
    "in_venv": sys.prefix != sys.base_prefix,
    "notebook_code_sha256": hashlib.sha256("\n".join(
        cell.get("source", []) if isinstance(cell.get("source", []), str) else "".join(cell.get("source", []))
        for cell in json.loads((ROOT / "data_code/01_small_batch_probe.ipynb").read_text())["cells"]
        if cell["cell_type"] == "code").encode()).hexdigest(),
    "packages": {name: version(name) for name in
                 ["numpy", "pandas", "requests", "yfinance", "python-dotenv", "ipykernel", "nbformat", "nbclient"]},
}
print("Repository:", ROOT)
print("Kernel Python:", sys.executable)
print("Environment:", ENVIRONMENT)
print("FRED_API_KEY present:", bool(FRED_KEY))
print("Run ID:", RUN_ID, "| Expected calendar days:", len(EXPECTED_DAYS))
if not (sys.implementation.name == "cpython" and sys.version_info[:2] == (3, 14)
        and platform.system() == "Darwin" and platform.machine() == "arm64"
        and Path(sys.prefix) == ROOT / ".venv"):
    raise RuntimeError("approved_local_runtime_mismatch")

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def safe_text(value):
    text = str(value)
    for secret in _SECRET_VALUES:
        text = text.replace(secret, "[REDACTED]").replace(quote(secret, safe=""), "[REDACTED]")
    text = re.sub(r"(?i)((?:api_key|apikey|token|crumb|authorization|cookie)=)[^&\s\"<>]+", r"\1[REDACTED]", text)
    return text

def safe_object(value):
    if isinstance(value, dict):
        return {str(k): ("[REDACTED]" if re.fullmatch(r"(?i)(api_?key|token|crumb|authorization|cookie|set-cookie)", str(k))
                        else safe_object(v)) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [safe_object(v) for v in value]
    if isinstance(value, str):
        return safe_text(value)
    if isinstance(value, np.generic):
        return safe_object(value.item())
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value

def save_bytes(relative, payload, kind):
    path = RUN_DIR / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    original_hash = hashlib.sha256(payload).hexdigest()
    for secret in _SECRET_VALUES:
        payload = payload.replace(secret.encode(), b"[REDACTED]")
        payload = payload.replace(quote(secret, safe="").encode(), b"[REDACTED]")
    with path.open("xb") as stream:
        stream.write(payload)
    ARTIFACTS.append(dict(path=str(path.relative_to(ROOT)), kind=kind, bytes=len(payload),
                          sha256=hashlib.sha256(payload).hexdigest(),
                          secret_redaction_applied=hashlib.sha256(payload).hexdigest() != original_hash))
    return str(path.relative_to(ROOT))

def save_json(relative, value, kind="system_metadata"):
    payload = json.dumps(safe_object(value), ensure_ascii=False, indent=2, default=str, allow_nan=False)
    return save_bytes(relative, payload.encode(), kind)

def save_csv(relative, frame, kind="validation_table", index=False):
    return save_bytes(relative, frame.to_csv(index=index).encode("utf-8"), kind)

def record_request(item):
    item = safe_object(item)
    REQUESTS.append(item)
    path = RUN_DIR / "system" / "requests.jsonl"
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a") as stream:
        stream.write(json.dumps(item, ensure_ascii=False, default=str) + "\n")

def http_get(source, dataset, endpoint, params=None, stem="response", suffix="json"):
    public_params = {k: v for k, v in (params or {}).items()
                     if not re.search(r"(?i)(key|token|crumb|authorization)", k)}
    for attempt in range(1, MAX_ATTEMPTS + 1):
        item = dict(source=source, dataset_id=dataset, endpoint=endpoint,
                    parameters=public_params, attempt=attempt, requested_at=utc_now(),
                    timeout_seconds=list(HTTP_TIMEOUT), authentication="environment_key" if source == "FRED" else "none")
        try:
            response = SESSION.get(endpoint, params=params, timeout=HTTP_TIMEOUT, allow_redirects=False)
            item.update(http_status=response.status_code, retrieved_at=utc_now(),
                        request_status="success" if response.status_code == 200 else "failed",
                        response_headers={k: response.headers[k] for k in
                                          ["Content-Type", "Date", "ETag", "Last-Modified", "Retry-After"]
                                          if k in response.headers})
            extension = suffix if response.status_code == 200 else "body"
            item["artifact_path"] = save_bytes(
                f"raw/{source.lower()}/{stem}_attempt{attempt}.{extension}",
                response.content, "http_response_body")
            item["raw_row_count"] = None
            record_request(item)
            if response.status_code == 200:
                return response, item
            # No bypass or unbounded retries for authentication, blocks, or redirects.
            if response.status_code not in (429, 500, 502, 503, 504):
                return response, item
            retry_after = response.headers.get("Retry-After", "")
            if retry_after.isdigit() and int(retry_after) > 15:
                return response, item
            delay = max(2 ** attempt, int(retry_after) if retry_after.isdigit() else 0)
        except requests.RequestException as exc:
            item.update(http_status=None, retrieved_at=utc_now(), request_status="failed",
                        error_type=type(exc).__name__, artifact_path=None)
            record_request(item)
            response = None
            delay = 2 ** attempt
        if attempt < MAX_ATTEMPTS:
            time.sleep(delay)
    return response, item

def base_result(source, dataset):
    return dict(source=source, dataset_id=dataset, request_status="not_requested", http_status=None,
                parse_status="not_run", raw_row_count=None, filtered_row_count=None,
                observed_start=None, observed_end=None, missing_count=None, duplicate_count=None,
                unit="unverified", unit_verification_status="unverified", validation_status="not_run",
                warnings=[], retrieved_at=None, artifact_path=None)

def finish(result):
    RESULTS.append(result)
    # Checkpoint each dataset so another source failure cannot erase it.
    save_json(f"system/results/{len(RESULTS):02d}_{result['source']}_{re.sub('[^A-Za-z0-9_-]', '_', result['dataset_id'])}.json", result)
    print(result["source"], result["dataset_id"], result["request_status"], result["parse_status"],
          result["validation_status"], "rows:", result["filtered_row_count"])

def date_stats(frame, date_column, value_columns):
    dates = pd.to_datetime(frame[date_column], errors="coerce")
    return dict(filtered_row_count=len(frame),
                observed_start=dates.min().date().isoformat() if dates.notna().any() else None,
                observed_end=dates.max().date().isoformat() if dates.notna().any() else None,
                duplicate_count=int(dates.duplicated().sum()),
                invalid_date_count=int(dates.isna().sum()),
                missing_count=int(frame[value_columns].isna().sum().sum()),
                dates_sorted=bool(dates.is_monotonic_increasing))

def quality_status(result, violations):
    result["violations"] = violations
    if result.get("filtered_row_count", 0) == 0 or any(violations.values()):
        result["validation_status"] = "failed"
    elif result.get("missing_count") or result["warnings"] or result["unit_verification_status"] == "unverified":
        result["validation_status"] = "warning"
    else:
        result["validation_status"] = "passed"

def error_result(result, exc):
    result["validation_status"] = "not_run" if result["parse_status"] == "not_run" else "failed"
    result["warnings"].append("exception_type:" + type(exc).__name__)
    if result["request_status"] == "not_requested":
        result["request_status"] = "failed"
    finish(result)

save_json("system/run-start.json", dict(run_id=RUN_ID, started_at=utc_now(), environment=ENVIRONMENT,
          period=dict(start=str(START.date()), end=str(END.date()), inclusive="both"),
          regions=REGIONS, nasa_time_standard="UTC", weather_variables=WEATHER_VARS,
          sources=["Yahoo", "FRED", "NASA", "CFTC"], cftc_archive_years=[2023, 2024, 2025],
          not_requested=["GSCPI"], fred_key_present=bool(FRED_KEY), adr="ADR-003",
          raw_definition="requests response entity bytes; yfinance returns stored separately, not HTTP raw",
          availability_policy="published_at/available_at unknown unless directly evidenced"))

Repository: /Users/sj/MyDrive/MyWorkspace/Projects/Coffee_Price_Prediction
Kernel Python: /Users/sj/MyDrive/MyWorkspace/Projects/Coffee_Price_Prediction/.venv/bin/python
Environment: {'python_version': '3.14.7', 'python_executable': '/Users/sj/MyDrive/MyWorkspace/Projects/Coffee_Price_Prediction/.venv/bin/python', 'implementation': 'cpython', 'platform': 'macOS-27.0-arm64-arm-64bit-Mach-O', 'machine': 'arm64', 'in_venv': True, 'notebook_code_sha256': '1c663856397584724f6a534e173a856cbaf71b12a14c815a2470d633ef6cccbf', 'packages': {'numpy': '2.5.3', 'pandas': '3.0.5', 'requests': '2.34.2', 'yfinance': '1.7.0', 'python-dotenv': '1.2.3', 'ipykernel': '7.3.0', 'nbformat': '5.11.1', 'nbclient': '0.11.0'}}
FRED_API_KEY present: True
Run ID: 20260912T063021795259Z_99d695f0 | Expected calendar days: 732


'data/raw/probes/20260912T063021795259Z_99d695f0/system/run-start.json'

# 2. Yahoo Finance — market_daily

KC=F·BRL=X, 일별, auto_adjust=False, back_adjust=False, repair=False,
actions=False, keepna=True, rounding=False. end가 제외되는 인터페이스에는 2026-01-01을 전달한 뒤 승인 구간으로 필터링한다.
library_return의 CSV 및 pickle은 **가공 전 yfinance 반환 테이블**이며 HTTP 원본 응답이 아니다.
HTTP 상태를 알 수 없으면 null로 둔다. 가격은 변환하지 않는다.

[ICE Coffee C 명세](https://www.ice.com/products/15/Coffee-C-Futures)는 센트/lb 표시를 설명한다.
Yahoo metadata의 currency=USD만으로 Yahoo 수치의 배율을 확정하지 않는다.
BRL=X도 provider 이름/통화로 방향을 확인하지 못하면 unverified다.
환율 Volume 0/결측은 진단 정보이며 선물 거래량 오류와 동일하게 판정하지 않는다.

In [2]:
def probe_yahoo(symbol):
    result = base_result("Yahoo", symbol)
    slug = symbol.replace("=", "_")
    try:
        ticker = yf.Ticker(symbol)
        frame = None
        options = dict(start=str(START.date()), end=str((END + pd.Timedelta(days=1)).date()),
                       interval="1d", auto_adjust=False, back_adjust=False, repair=False,
                       actions=False, keepna=True, rounding=False, timeout=30, raise_errors=True)
        for attempt in range(1, MAX_ATTEMPTS + 1):
            try:
                # Library logs may contain internal session data; do not persist them.
                with contextlib.redirect_stdout(StringIO()), contextlib.redirect_stderr(StringIO()):
                    frame = ticker.history(**options)
                record_request(dict(source="Yahoo", dataset_id=symbol, interface="yfinance.Ticker.history",
                                    parameters=options, attempt=attempt, http_status=None,
                                    request_status="success", retrieved_at=utc_now()))
                break
            except Exception as exc:
                record_request(dict(source="Yahoo", dataset_id=symbol, interface="yfinance.Ticker.history",
                                    parameters=options, attempt=attempt, http_status=None,
                                    request_status="failed", retrieved_at=utc_now(), error_type=type(exc).__name__))
                if "RateLimit" in type(exc).__name__ or attempt == MAX_ATTEMPTS:
                    raise
                # Retry only transport timeouts/connections, not unknown authorization/schema errors.
                if not any(word in type(exc).__name__ for word in ("Timeout", "Connection")):
                    raise
                time.sleep(2 ** attempt)
        result.update(request_status="success", retrieved_at=utc_now(), raw_row_count=len(frame))
        result["library_return_path"] = save_csv(f"library_return/yahoo/{slug}.csv", frame, "library_return", index=True)
        buffer = BytesIO()
        frame.to_pickle(buffer)
        save_bytes(f"library_return/yahoo/{slug}.pkl", buffer.getvalue(), "library_return")
        metadata = {}
        try:
            with contextlib.redirect_stdout(StringIO()), contextlib.redirect_stderr(StringIO()):
                metadata["history_metadata"] = ticker.get_history_metadata()
                metadata["quote_metadata"] = ticker.get_info()
            result["metadata_status"] = "success"
        except Exception as exc:
            result["metadata_status"] = "partial_or_failed"
            result["warnings"].append("metadata:" + type(exc).__name__)
        save_json(f"library_return/yahoo/{slug}_metadata.json", metadata, "library_return_metadata")
        columns = ["Open", "High", "Low", "Close"]
        if not set(columns).issubset(frame.columns):
            raise ValueError("missing_ohlc_columns")
        checked = frame.copy()
        checked.insert(0, "observation_date", pd.to_datetime(frame.index.strftime("%Y-%m-%d")))
        checked = checked.loc[checked["observation_date"].between(START, END)].copy()
        for name in columns + ["Volume"]:
            checked[name] = pd.to_numeric(checked[name], errors="coerce") if name in checked else np.nan
        checked["symbol"] = symbol
        checked["retrieved_at"] = result["retrieved_at"]
        checked["published_at"] = None
        checked["available_at"] = None
        result.update(parse_status="success", **date_stats(checked, "observation_date", columns + ["Volume"]))
        result["volume_zero_count"] = int(checked["Volume"].eq(0).sum())
        result["volume_missing_count"] = int(checked["Volume"].isna().sum())
        result["provider_index_timezone"] = str(frame.index.tz)
        info = metadata.get("quote_metadata", {})
        hist = metadata.get("history_metadata", {})
        name = " ".join(str(v) for v in [info.get("shortName", ""), info.get("longName", ""),
                                        hist.get("shortName", ""), hist.get("longName", "")])
        currency = info.get("currency", hist.get("currency"))
        result["provider_currency"] = currency
        result["provider_name"] = name
        if symbol == "KC=F":
            result["contract_specification_unit"] = "US cents/lb"
            result["contract_specification_source"] = "https://www.ice.com/products/15/Coffee-C-Futures"
            if currency in ("USX", "USC"):
                result.update(unit="US cents/lb", unit_verification_status="provider_subunit_and_ICE_specification")
            else:
                result["warnings"].append("Yahoo price scale unverified; ICE unit alone and currency USD do not prove scale")
            result["warnings"].append("KC=F contract connection and roll methodology unverified; no roll adjustment")
        elif re.search(r"USD\s*/\s*BRL", name, re.I) and currency == "BRL":
            result.update(unit="BRL per USD", unit_verification_status="provider_pair_name_and_currency")
        else:
            result["warnings"].append("FX direction unverified from provider metadata")
        valid = checked[columns].notna().all(axis=1) & np.isfinite(checked[columns]).all(axis=1)
        violations = dict(
            invalid_dates=result["invalid_date_count"], duplicate_dates=result["duplicate_count"],
            high_below_low=int((valid & checked.High.lt(checked.Low)).sum()),
            open_outside_range=int((valid & (checked.Open.lt(checked.Low) | checked.Open.gt(checked.High))).sum()),
            close_outside_range=int((valid & (checked.Close.lt(checked.Low) | checked.Close.gt(checked.High))).sum()),
            nonfinite_ohlc=int((checked[columns].notna() & ~np.isfinite(checked[columns])).sum().sum()))
        if symbol == "BRL=X":
            result["warnings"].append("FX volume zero/missing is diagnostic, not futures-volume quality failure")
        result["artifact_path"] = save_csv(f"validation/market_{slug}.csv", checked, index=False)
        quality_status(result, violations)
        finish(result)
    except Exception as exc:
        error_result(result, exc)

In [3]:
probe_yahoo("KC=F")

Yahoo KC=F success success failed rows: 506


In [4]:
probe_yahoo("BRL=X")

Yahoo BRL=X success success failed rows: 523


# 3. FRED — macro_observations

DFF·DTWEXBGS·DCOILWTICO의 observations와 series metadata를 각각 조회한다.
raw JSON의 "."와 realtime 구간을 보존하고, 검증 표에서만 "."를 결측으로 파싱한다.
frequency·units·seasonal_adjustment를 보존한다. 음수에 일괄 오류 규칙을 적용하지 않는다.
realtime 필드는 조회·수정 상태에 관한 필드이며 각 관측치의 실제 발표시각으로 바꾸지 않는다.
현재 조회이며 과거 vintage 재현이 아니다. GSCPI는 요청하지 않는다.

In [5]:
def probe_fred(series_id):
    result = base_result("FRED", series_id)
    if not FRED_KEY:
        result.update(request_status="auth_missing", warnings=["FRED_API_KEY not configured"])
        finish(result)
        return
    try:
        common = dict(series_id=series_id, api_key=FRED_KEY, file_type="json")
        observation_response, observation_request = http_get(
            "FRED", series_id, "https://api.stlouisfed.org/fred/series/observations",
            dict(common, observation_start=str(START.date()), observation_end=str(END.date()),
                 sort_order="asc", limit=100000), stem=series_id + "_observations")
        metadata_response, metadata_request = http_get(
            "FRED", series_id, "https://api.stlouisfed.org/fred/series",
            common, stem=series_id + "_series")
        result.update(http_status=observation_request["http_status"],
                      metadata_http_status=metadata_request["http_status"],
                      retrieved_at=observation_request["retrieved_at"],
                      request_status=observation_request["request_status"],
                      raw_artifact_path=observation_request.get("artifact_path"))
        if observation_response is None or observation_response.status_code != 200:
            result["warnings"].append("observations_request_failed")
            finish(result)
            return
        payload = observation_response.json()
        if "observations" not in payload:
            raise ValueError("observations_payload_missing")
        raw = pd.DataFrame(payload["observations"])
        result["raw_row_count"] = len(raw)
        meta = {}
        if metadata_response is not None and metadata_response.status_code == 200:
            metadata_payload = metadata_response.json()
            entries = metadata_payload.get("seriess", [])
            meta = next((entry for entry in entries if entry.get("id") == series_id), {})
        if not meta:
            result["request_status"] = "partial"
            result["warnings"].append("series_metadata_missing")
        save_json(f"system/fred_{series_id}_metadata.json", meta)
        if not {"date", "value"}.issubset(raw.columns):
            raise ValueError("fred_schema_missing")
        checked = raw.rename(columns={"date": "observation_date"}).copy()
        checked["observation_date"] = pd.to_datetime(checked["observation_date"], errors="coerce")
        result["raw_invalid_date_count"] = int(checked["observation_date"].isna().sum())
        checked = checked.loc[checked["observation_date"].between(START, END)].copy()
        result["sentinel_count"] = int(checked["value"].eq(".").sum())
        original_values = checked["value"].copy()
        checked["value"] = pd.to_numeric(checked["value"].replace(".", np.nan), errors="coerce")
        invalid_numeric = int((original_values.ne(".") & checked["value"].isna()).sum())
        for column in ("units", "frequency", "frequency_short", "seasonal_adjustment", "seasonal_adjustment_short"):
            checked[column] = meta.get(column)
        checked["series_id"] = series_id
        checked["retrieved_at"] = result["retrieved_at"]
        checked["published_at"] = None
        checked["available_at"] = None
        checked["availability_basis"] = "unknown"
        result.update(parse_status="success", **date_stats(checked, "observation_date", ["value"]))
        if meta.get("units"):
            result.update(unit=meta["units"], unit_verification_status="verified_series_metadata")
        result["frequency"] = meta.get("frequency")
        result["seasonal_adjustment"] = meta.get("seasonal_adjustment")
        result["negative_value_count"] = int(checked["value"].lt(0).sum())
        result["warnings"].append("current observation snapshot; original publication times/vintages not reconstructed")
        if series_id == "DFF":
            result["missing_expected_calendar_dates"] = len(EXPECTED_DAYS.difference(checked.observation_date))
        result["artifact_path"] = save_csv(f"validation/macro_{series_id}.csv", checked)
        quality_status(result, dict(invalid_dates=result["raw_invalid_date_count"],
                                    duplicate_dates=result["duplicate_count"], invalid_numbers=invalid_numeric))
        finish(result)
    except Exception as exc:
        error_result(result, exc)

In [6]:
probe_fred("DFF")

FRED DFF success success warning rows: 732


In [7]:
probe_fred("DTWEXBGS")

FRED DTWEXBGS success success warning rows: 523


In [8]:
probe_fred("DCOILWTICO")

FRED DCOILWTICO success success warning rows: 523


# 4. NASA POWER — weather_daily

ADR-003의 6개 승인 좌표·5개 변수, community=AG, UTC Daily, JSON.
승인된 732일 구간을 지점별 1회 요청한다(실패 시 제한 retry). 별도 기간 분할은 하지 않는다.
header.fill_value를 우선 사용한다. 선언된 sentinel만 검증 표에서 결측으로 파싱한다.
음수 강수, RH2M 0–100, 유효한 최저≤평균≤최고 관계를 진단하며 원값은 삭제하지 않는다.

geometry와 header를 보존하고 요청 좌표 재표시 여부를 비교한다.
응답에 명시된 원천 격자 정보가 없으면 grid_id/격자 중심은 unknown이다.
지점 간 비교는 가격과의 관계 EDA가 아니라 기상 자료 중복 진단이다.

In [9]:
def probe_nasa(region):
    region_id = region["region_id"]
    result = base_result("NASA", region_id)
    try:
        params = dict(parameters=",".join(WEATHER_VARS), community="AG",
                      longitude=region["longitude"], latitude=region["latitude"],
                      start=START.strftime("%Y%m%d"), end=END.strftime("%Y%m%d"),
                      format="JSON", **{"time-standard": "UTC"})
        response, request = http_get("NASA", region_id, "https://power.larc.nasa.gov/api/temporal/daily/point",
                                     params, stem=region_id)
        result.update(request_status=request["request_status"], http_status=request["http_status"],
                      retrieved_at=request["retrieved_at"], raw_artifact_path=request.get("artifact_path"))
        if response is None or response.status_code != 200:
            result["warnings"].append("NASA request failed; no fallback")
            finish(result)
            return
        duplicate_json_keys = []
        def unique_pairs(pairs):
            obj = {}
            for key, value in pairs:
                if key in obj:
                    duplicate_json_keys.append(key)
                obj[key] = value
            return obj
        payload = json.loads(response.content, object_pairs_hook=unique_pairs)
        header = payload.get("header", {})
        parameter_meta = payload.get("parameters", {})
        geometry = payload.get("geometry", {})
        values = payload.get("properties", {}).get("parameter", {})
        messages = payload.get("messages", [])
        result["payload_messages"] = safe_object(messages)
        result["response_time_standard"] = header.get("time_standard")
        result["fill_value"] = header.get("fill_value")
        result["units_by_variable"] = {key: parameter_meta.get(key, {}).get("units") for key in WEATHER_VARS}
        coordinates = geometry.get("coordinates", [])
        echoes_request = (len(coordinates) >= 2 and
                          np.isclose(coordinates[0], region["longitude"]) and
                          np.isclose(coordinates[1], region["latitude"]))
        result["geometry_interpretation"] = "matches_requested_point; not proof of native grid center" if echoes_request else "unknown; see response geometry"
        result["grid_id"] = None
        save_json(f"system/nasa_{region_id}_metadata.json",
                  dict(requested_location=region, geometry=geometry, header=header,
                       parameters=parameter_meta, messages=messages,
                       geometry_interpretation=result["geometry_interpretation"], grid_id=None))
        missing_vars = sorted(set(WEATHER_VARS) - set(values))
        if missing_vars:
            result["warnings"].append("missing_variables:" + ",".join(missing_vars))
            raise ValueError("NASA_variables_missing")
        raw = pd.DataFrame({key: values[key] for key in WEATHER_VARS})
        result["raw_row_count"] = len(raw)
        parsed_dates = pd.to_datetime(raw.index, format="%Y%m%d", errors="coerce")
        result["raw_invalid_date_count"] = int(parsed_dates.isna().sum())
        raw.insert(0, "observation_date", parsed_dates)
        checked = raw.loc[raw["observation_date"].between(START, END)].copy().reset_index(drop=True)
        sentinel_counts, invalid_numeric = {}, 0
        for variable in WEATHER_VARS:
            original = checked[variable]
            numeric = pd.to_numeric(original, errors="coerce")
            invalid_numeric += int((original.notna() & numeric.isna()).sum())
            fill = parameter_meta.get(variable, {}).get("fill_value", header.get("fill_value"))
            if fill is None:
                sentinel_counts[variable] = None
                result["warnings"].append(variable + ": fill value not declared; no guessed replacement")
            else:
                sentinel_counts[variable] = int(numeric.eq(float(fill)).sum())
                numeric = numeric.mask(numeric.eq(float(fill)))
            checked[variable] = numeric
        result["sentinel_counts"] = sentinel_counts
        checked["region_id"] = region_id
        checked["requested_latitude"] = region["latitude"]
        checked["requested_longitude"] = region["longitude"]
        checked["aggregation_time_standard"] = header.get("time_standard")
        checked["retrieved_at"] = result["retrieved_at"]
        checked["published_at"] = None
        checked["available_at"] = None
        checked["availability_basis"] = "unknown"
        result.update(parse_status="success", **date_stats(checked, "observation_date", WEATHER_VARS))
        missing_dates = EXPECTED_DAYS.difference(checked.observation_date)
        result["missing_expected_calendar_dates"] = len(missing_dates)
        result["missing_dates"] = [str(day.date()) for day in missing_dates]
        if all(result["units_by_variable"].values()):
            result.update(unit=json.dumps(result["units_by_variable"], sort_keys=True),
                          unit_verification_status="verified_response_metadata")
        else:
            result["warnings"].append("variable units missing")
        temp_valid = checked[["T2M_MIN", "T2M", "T2M_MAX"]].notna().all(axis=1)
        violations = dict(
            invalid_dates=result["raw_invalid_date_count"], duplicate_dates=result["duplicate_count"],
            missing_dates=len(missing_dates), duplicate_json_keys=len(duplicate_json_keys),
            invalid_numbers=invalid_numeric,
            negative_precipitation=int(checked.PRECTOTCORR.lt(0).sum()),
            humidity_out_of_range=int((checked.RH2M.lt(0) | checked.RH2M.gt(100)).sum()),
            temperature_order=int((temp_valid & (checked.T2M_MIN.gt(checked.T2M) | checked.T2M.gt(checked.T2M_MAX))).sum()),
            nonfinite_values=int((checked[WEATHER_VARS].notna() & ~np.isfinite(checked[WEATHER_VARS])).sum().sum()),
            wrong_time_standard=int(str(header.get("time_standard", "")).upper() != "UTC"))
        if messages:
            result["warnings"].append("response messages present; retained for review")
        result["warnings"].append("retrospective current weather snapshot; historical available_at unknown")
        result["artifact_path"] = save_csv(f"validation/weather_{region_id}.csv", checked)
        WEATHER_TABLES[region_id] = checked
        quality_status(result, violations)
        finish(result)
    except Exception as exc:
        error_result(result, exc)

In [10]:
probe_nasa(REGIONS[0])

NASA br_sul_minas success success warning rows: 732


In [11]:
probe_nasa(REGIONS[1])

NASA br_cerrado success success warning rows: 732


In [12]:
probe_nasa(REGIONS[2])

NASA br_alta_mogiana success success warning rows: 732


In [13]:
probe_nasa(REGIONS[3])

NASA co_huila success success warning rows: 732


In [14]:
probe_nasa(REGIONS[4])

NASA co_caldas success success warning rows: 732


In [15]:
probe_nasa(REGIONS[5])

NASA co_antioquia success success warning rows: 732


In [16]:
pairwise_rows = []
for left, right in combinations([region["region_id"] for region in REGIONS], 2):
    for variable in WEATHER_VARS:
        row = dict(left_region=left, right_region=right, variable=variable,
                   comparison_status="not_available", common_valid_count=0,
                   exact_on_common_valid=None, missing_pattern_equal=None,
                   complete_nonmissing_match=None, correlation=None)
        if left in WEATHER_TABLES and right in WEATHER_TABLES:
            a = WEATHER_TABLES[left].set_index("observation_date")
            b = WEATHER_TABLES[right].set_index("observation_date")
            if not a.index.has_duplicates and not b.index.has_duplicates:
                a, b = a[variable].reindex(EXPECTED_DAYS), b[variable].reindex(EXPECTED_DAYS)
                valid = a.notna() & b.notna() & np.isfinite(a) & np.isfinite(b)
                n = int(valid.sum())
                row.update(comparison_status="compared" if n else "no_common_valid_data", common_valid_count=n,
                           missing_pattern_equal=bool(a.isna().equals(b.isna())))
                if n:
                    equal = bool(np.array_equal(a[valid].to_numpy(), b[valid].to_numpy()))
                    row.update(exact_on_common_valid=equal,
                               complete_nonmissing_match=bool(equal and n == len(EXPECTED_DAYS)))
                if n >= 3 and a[valid].nunique() > 1 and b[valid].nunique() > 1:
                    correlation = float(a[valid].corr(b[valid]))
                    row["correlation"] = correlation if np.isfinite(correlation) else None
        pairwise_rows.append(row)
PAIRWISE = pd.DataFrame(pairwise_rows)
save_csv("validation/weather_pairwise.csv", PAIRWISE)
suspected_duplicate_pairs = []
for (left, right), group in PAIRWISE.groupby(["left_region", "right_region"]):
    if len(group) == len(WEATHER_VARS) and group["complete_nonmissing_match"].eq(True).all():
        suspected_duplicate_pairs.append([left, right])
save_json("system/weather_comparison.json",
          dict(expected_region_pairs=15, variable_comparisons=len(PAIRWISE),
               suspected_duplicate_pairs=suspected_duplicate_pairs,
               interpretation="exact complete series can suggest duplicate grid; correlation alone cannot identify grids"))
print("Weather pair-variable comparisons:", len(PAIRWISE))
print("Suspected duplicate pairs:", suspected_duplicate_pairs)
display(PAIRWISE)

Weather pair-variable comparisons: 75
Suspected duplicate pairs: []


,left_region,right_region,variable,comparison_status,common_valid_count,exact_on_common_valid,missing_pattern_equal,complete_nonmissing_match,correlation
0,br_sul_minas,br_cerrado,PRECTOTCORR,compared,732,False,True,False,0.494577
1,br_sul_minas,br_cerrado,T2M,compared,732,False,True,False,0.875502
2,br_sul_minas,br_cerrado,T2M_MIN,compared,732,False,True,False,0.894807
3,br_sul_minas,br_cerrado,T2M_MAX,compared,732,False,True,False,0.780554
4,br_sul_minas,br_cerrado,RH2M,compared,732,False,True,False,0.850339
...,...,...,...,...,...,...,...,...,...
70,co_caldas,co_antioquia,PRECTOTCORR,compared,732,False,True,False,0.568488
71,co_caldas,co_antioquia,T2M,compared,732,False,True,False,0.849312
72,co_caldas,co_antioquia,T2M_MIN,compared,732,False,True,False,0.477431
73,co_caldas,co_antioquia,T2M_MAX,compared,732,False,True,False,0.820770


# 5. CFTC — cot_weekly

보고서: **Disaggregated Futures Only**, 시장 문자열 **083731**.
공식 연간 ZIP은 2023·2024·2025의 전체 시장을 포함한다.
원본 ZIP/그 안의 텍스트를 보존하고, 검증 표만 report_date와 시장으로 필터링한다.
2023 연간 파일에서 승인 기간에 해당하는 행이 0개일 수 있으나 전체 Probe 결과 0행은 실패다.
Combined/Legacy나 다른 공급원으로 대체하지 않는다.

[공식 아카이브](https://www.cftc.gov/MarketReports/CommitmentsofTraders/HistoricalCompressed/index.htm),
[포지션·Open Interest 정의](https://www.cftc.gov/MarketReports/CommitmentsofTraders/ExplanatoryNotes/index.htm).
report_date는 보고 기준일이다. 공개일·available_at을 임의 생성하지 않는다.

In [17]:
CFTC_URLS = {year: f"https://www.cftc.gov/files/dea/history/fut_disagg_txt_{year}.zip"
             for year in (2023, 2024, 2025)}

def probe_cftc_year(year):
    entry = dict(year=year, report_type="Disaggregated Futures Only", raw_row_count=None,
                 coffee_row_count=None, filtered_row_count=None, parse_status="not_run")
    try:
        response, request = http_get("CFTC", "083731", CFTC_URLS[year], stem=f"disaggregated_futures_only_{year}", suffix="zip")
        entry.update(request)
        if response is None or response.status_code != 200:
            COT_ARCHIVES.append(entry)
            print("CFTC", year, "request failed", request["http_status"])
            return
        with ZipFile(BytesIO(response.content)) as archive:
            names = [name for name in archive.namelist() if name.lower().endswith((".txt", ".csv"))]
            if len(names) != 1:
                raise ValueError("unexpected_archive_members")
            info = archive.getinfo(names[0])
            if info.file_size > 250_000_000:
                raise ValueError("archive_uncompressed_size_limit")
            content = archive.read(names[0])
        entry["archive_member"] = names[0]
        entry["member_artifact_path"] = save_bytes(f"raw/cftc/disaggregated_futures_only_{year}_member.txt", content, "archive_member")
        try:
            text = content.decode("utf-8-sig")
            entry["encoding"] = "utf-8-sig"
        except UnicodeDecodeError:
            text = content.decode("cp1252")
            entry["encoding"] = "cp1252"
        raw = pd.read_csv(StringIO(text), dtype="string", keep_default_na=False)
        entry["raw_row_count"] = len(raw)
        entry["original_columns"] = list(raw.columns)
        aliases = {
            "market_code": ["CFTC_Contract_Market_Code"],
            "market_name": ["Market_and_Exchange_Names"],
            "report_date": ["Report_Date_as_YYYY-MM-DD", "Report_Date_as_MM_DD_YYYY"],
            "open_interest": ["Open_Interest_All"],
            "managed_money_long": ["M_Money_Positions_Long_All"],
            "managed_money_short": ["M_Money_Positions_Short_All"],
        }
        mapping = {}
        for canonical, candidates in aliases.items():
            matches = [column for column in raw.columns if column.strip() in candidates]
            if len(matches) != 1:
                raise ValueError("CFTC_required_column_missing_or_ambiguous")
            mapping[canonical] = matches[0]
        entry["column_mapping"] = mapping
        coffee = raw.loc[raw[mapping["market_code"]].str.strip().eq("083731")].copy()
        entry["coffee_row_count"] = len(coffee)
        checked = coffee[list(mapping.values())].rename(columns={v: k for k, v in mapping.items()}).copy()
        checked["market_code"] = checked["market_code"].str.strip()
        date_format = "%Y-%m-%d" if mapping["report_date"].endswith("YYYY-MM-DD") else "%m/%d/%Y"
        checked["report_date"] = pd.to_datetime(checked["report_date"], format=date_format, errors="coerce")
        entry["invalid_date_count"] = int(checked["report_date"].isna().sum())
        entry["raw_coffee_start"] = str(checked["report_date"].min().date()) if checked["report_date"].notna().any() else None
        entry["raw_coffee_end"] = str(checked["report_date"].max().date()) if checked["report_date"].notna().any() else None
        checked = checked.loc[checked["report_date"].between(START, END)].copy()
        entry["invalid_numeric_count"] = 0
        for column in ["open_interest", "managed_money_long", "managed_money_short"]:
            strings = checked[column].str.strip().str.replace(",", "", regex=False)
            numeric = pd.to_numeric(strings, errors="coerce")
            entry["invalid_numeric_count"] += int((strings.ne("") & numeric.isna()).sum())
            checked[column] = numeric
        checked["report_type"] = "Disaggregated Futures Only"
        checked["archive_year"] = year
        checked["retrieved_at"] = request["retrieved_at"]
        checked["published_at"] = None
        checked["available_at"] = None
        checked["availability_basis"] = "unknown"
        entry.update(parse_status="success", filtered_row_count=len(checked))
        COT_PARTS.append(checked)
        COT_ARCHIVES.append(entry)
        print("CFTC", year, "archive rows:",len(raw),"Coffee rows:",len(coffee),"approved-period rows:",len(checked))
    except Exception as exc:
        entry.update(parse_status="failed", error_type=type(exc).__name__)
        COT_ARCHIVES.append(entry)
        print("CFTC", year, "parse/request failed:",type(exc).__name__)

In [18]:
probe_cftc_year(2023)

CFTC 2023 archive rows: 12534 Coffee rows: 52 approved-period rows: 0


In [19]:
probe_cftc_year(2024)

CFTC 2024 archive rows: 13601 Coffee rows: 53 approved-period rows: 53


In [20]:
probe_cftc_year(2025)

CFTC 2025 archive rows: 13630 Coffee rows: 52 approved-period rows: 52


In [21]:
save_json("system/cftc_archive_inventory.json", COT_ARCHIVES)
cot_result = base_result("CFTC", "083731")
cot_result["http_status"] = [entry.get("http_status") for entry in COT_ARCHIVES]
cot_result["retrieved_at"] = max((entry.get("retrieved_at", "") for entry in COT_ARCHIVES), default=None)
cot_result["raw_row_count"] = sum(entry["raw_row_count"] for entry in COT_ARCHIVES if entry.get("raw_row_count") is not None) if COT_PARTS else None
cot_result["raw_count_scope"] = "all markets across successfully parsed yearly archives"
cot_result["coffee_raw_row_count"] = sum(entry.get("coffee_row_count") or 0 for entry in COT_ARCHIVES)
cot_result["request_status"] = ("success" if all(entry.get("request_status") == "success" for entry in COT_ARCHIVES)
                                else "partial" if COT_PARTS else "failed")
if COT_PARTS:
    cot = pd.concat(COT_PARTS, ignore_index=True)
    numeric_columns = ["open_interest", "managed_money_long", "managed_money_short"]
    cot_result.update(parse_status="success" if all(entry["parse_status"] == "success" for entry in COT_ARCHIVES) else "partial",
                      unit="contracts", unit_verification_status="official_CFTC_report_definition",
                      **date_stats(cot, "report_date", numeric_columns))
    cot_result["market_names"] = sorted(cot["market_name"].dropna().unique().tolist())
    cot_result["warnings"].append("report_date is not release date; publication/available_at not reconstructed")
    if len(COT_PARTS) != 3:
        cot_result["warnings"].append("not all three approved yearly archives parsed")
    cot_result["artifact_path"] = save_csv("validation/cot_083731.csv", cot)
    quality_status(cot_result, dict(
        duplicate_dates=cot_result["duplicate_count"],
        invalid_dates=sum(entry.get("invalid_date_count", 0) for entry in COT_ARCHIVES),
        invalid_numbers=sum(entry.get("invalid_numeric_count", 0) for entry in COT_ARCHIVES),
        missing_numbers=int(cot[numeric_columns].isna().sum().sum()),
        negative_positions=int(cot[numeric_columns].lt(0).sum().sum()),
        wrong_market_code=int(cot["market_code"].ne("083731").sum()),
        unexpected_market_name=int(not bool(cot["market_name"].str.contains("COFFEE", case=False).all()))))
else:
    cot_result.update(parse_status="failed" if any(entry["parse_status"] == "failed" for entry in COT_ARCHIVES) else "not_run", validation_status="not_run", warnings=["No successfully parsed Coffee C archive; no replacement source"])
finish(cot_result)

CFTC 083731 success success warning rows: 105


# 6. 저장·종합 검증

raw/: requests 응답 본문 및 CFTC 전체 시장 아카이브.
library_return/: 가공 전 yfinance 반환 표와 라이브러리 metadata.
validation/: 승인 구간의 정합성 검사 CSV(가격 변환·보간·행 삭제 없음).
system/: 비밀값을 제외한 요청 이력·메타데이터·검증 요약·manifest.

request_status, parse_status, validation_status를 분리한다.
missing_count는 각 데이터셋 핵심 값 컬럼의 결측 셀 수이며, 달력일 누락은 별도 필드다.
warning에는 단위/빈티지 한계도 포함되므로 관측값 물리적 위반과 구별한다.
available_at은 근거 없이 생성하지 않았다. CSV는 학습용 확정 데이터셋이 아니다.
각 실행 결과는 고유 run_id로 보존한다. 이 셀까지 실행한 뒤 notebook도 저장한다.

In [22]:
SUMMARY = pd.DataFrame(RESULTS)
summary_columns = ["source", "dataset_id", "request_status", "http_status", "parse_status",
                   "raw_row_count", "filtered_row_count", "observed_start", "observed_end",
                   "missing_count", "duplicate_count", "unit", "unit_verification_status",
                   "validation_status", "warnings", "retrieved_at", "artifact_path"]
summary_csv = SUMMARY.reindex(columns=summary_columns).copy()
for column in ("warnings", "http_status"):
    summary_csv[column] = summary_csv[column].map(lambda value: json.dumps(value, ensure_ascii=False, default=str))
save_csv("system/validation_summary.csv", summary_csv, kind="validation_summary")
save_json("system/validation_summary.json", RESULTS)
log_path = RUN_DIR / "system" / "requests.jsonl"
if log_path.exists():
    payload = log_path.read_bytes()
    ARTIFACTS.append(dict(path=str(log_path.relative_to(ROOT)), kind="request_log", bytes=len(payload),
                          sha256=hashlib.sha256(payload).hexdigest(), secret_redaction_applied=False))
manifest = dict(
    run_id=RUN_ID, completed_at=utc_now(), environment=ENVIRONMENT,
    period=dict(start=str(START.date()), end=str(END.date()), inclusive="both"),
    request_count=len(REQUESTS), dataset_count=len(RESULTS), expected_dataset_count=12,
    raw_artifact_definition="requests response entity bytes; yfinance is a library return, never HTTP raw",
    cftc_download_scope="full all-market yearly archives 2023/2024/2025; validation filters approved interval and 083731",
    region_policy="ADR-003 Probe only; unweighted; no station/grid representativeness claim",
    source_failures_isolated=True, mock_fallback=False, value_imputation=False,
    historical_vintage_reconstructed=False, artifacts=ARTIFACTS.copy())
save_json("system/manifest.json", manifest)
display(SUMMARY.reindex(columns=summary_columns))
print("RUN_DIR:", RUN_DIR)
print("All 12 datasets attempted:", len(RESULTS) == 12)
print("Request status counts:", SUMMARY["request_status"].value_counts().to_dict())
print("Validation status counts:", SUMMARY["validation_status"].value_counts().to_dict())
print("Results are a data access/integrity Probe, not a leakage-free training dataset.")

,source,dataset_id,request_status,http_status,parse_status,raw_row_count,filtered_row_count,observed_start,observed_end,missing_count,duplicate_count,unit,unit_verification_status,validation_status,warnings,retrieved_at,artifact_path
0,Yahoo,KC=F,success,None,success,506,506,2024-01-02,2025-12-31,8,0,US cents/lb,provider_subunit_and_ICE_specification,failed,[KC=F contract connection and roll methodology...,2026-09-12T06:30:23.227785+00:00,data/raw/probes/20260912T063021795259Z_99d695f...
1,Yahoo,BRL=X,success,None,success,523,523,2024-01-01,2025-12-31,16,0,BRL per USD,provider_pair_name_and_currency,failed,"[FX volume zero/missing is diagnostic, not fut...",2026-09-12T06:30:24.484203+00:00,data/raw/probes/20260912T063021795259Z_99d695f...
2,FRED,DFF,success,200,success,732,732,2023-12-31,2025-12-31,0,0,Percent,verified_series_metadata,warning,[current observation snapshot; original public...,2026-09-12T06:30:38.240512+00:00,data/raw/probes/20260912T063021795259Z_99d695f...
3,FRED,DTWEXBGS,success,200,success,523,523,2024-01-01,2025-12-31,22,0,Index Jan 2006=100,verified_series_metadata,warning,[current observation snapshot; original public...,2026-09-12T06:31:12.442571+00:00,data/raw/probes/20260912T063021795259Z_99d695f...
4,FRED,DCOILWTICO,success,200,success,523,523,2024-01-01,2025-12-31,25,0,Dollars per Barrel,verified_series_metadata,warning,[current observation snapshot; original public...,2026-09-12T06:31:21.829580+00:00,data/raw/probes/20260912T063021795259Z_99d695f...
5,NASA,br_sul_minas,success,200,success,732,732,2023-12-31,2025-12-31,0,0,"{""PRECTOTCORR"": ""mm/day"", ""RH2M"": ""%"", ""T2M"": ...",verified_response_metadata,warning,[retrospective current weather snapshot; histo...,2026-09-12T06:31:25.816237+00:00,data/raw/probes/20260912T063021795259Z_99d695f...
6,NASA,br_cerrado,success,200,success,732,732,2023-12-31,2025-12-31,0,0,"{""PRECTOTCORR"": ""mm/day"", ""RH2M"": ""%"", ""T2M"": ...",verified_response_metadata,warning,[retrospective current weather snapshot; histo...,2026-09-12T06:31:26.841038+00:00,data/raw/probes/20260912T063021795259Z_99d695f...
7,NASA,br_alta_mogiana,success,200,success,732,732,2023-12-31,2025-12-31,0,0,"{""PRECTOTCORR"": ""mm/day"", ""RH2M"": ""%"", ""T2M"": ...",verified_response_metadata,warning,[retrospective current weather snapshot; histo...,2026-09-12T06:31:27.749490+00:00,data/raw/probes/20260912T063021795259Z_99d695f...
8,NASA,co_huila,success,200,success,732,732,2023-12-31,2025-12-31,0,0,"{""PRECTOTCORR"": ""mm/day"", ""RH2M"": ""%"", ""T2M"": ...",verified_response_metadata,warning,[retrospective current weather snapshot; histo...,2026-09-12T06:31:30.138965+00:00,data/raw/probes/20260912T063021795259Z_99d695f...
9,NASA,co_caldas,success,200,success,732,732,2023-12-31,2025-12-31,0,0,"{""PRECTOTCORR"": ""mm/day"", ""RH2M"": ""%"", ""T2M"": ...",verified_response_metadata,warning,[retrospective current weather snapshot; histo...,2026-09-12T06:31:33.933775+00:00,data/raw/probes/20260912T063021795259Z_99d695f...


RUN_DIR: /Users/sj/MyDrive/MyWorkspace/Projects/Coffee_Price_Prediction/data/raw/probes/20260912T063021795259Z_99d695f0
All 12 datasets attempted: True
Request status counts: {'success': 12}
Validation status counts: {'warning': 10, 'failed': 2}
Results are a data access/integrity Probe, not a leakage-free training dataset.
